In [1]:
import pandas as pd
import numpy as np
import os
import pathlib

In [ ]:
# Load data from the tableau data
CURR_DIR_PATH = pathlib.Path(os.getcwd())
SSP_MODELING_DIR_PATH = CURR_DIR_PATH.parent
DATA_DIR_PATH = SSP_MODELING_DIR_PATH.joinpath("tableau/data/")
RUN_NAME = "2026-03-13T07;36;11.592365"

In [3]:
drivers = pd.read_csv(DATA_DIR_PATH.joinpath( "drivers_libya.csv"))
emissions = pd.read_csv(DATA_DIR_PATH.joinpath( f"decomposed_emissions_libya_2023_sisepuede_results_sisepuede_run_{RUN_NAME}_WIDE_INPUTS_OUTPUTS.csv"))


/var/folders/3p/n363h71s3gg0h1brsrv7f_zr0000gn/T/ipykernel_15973/2321402070.py:1: DtypeWarning: Columns (10,20) have mixed types. Specify dtype option on import or set low_memory=False.
  drivers = pd.read_csv(DATA_DIR_PATH.joinpath( "drivers_libya.csv"))


In [4]:
drivers['strategy'].unique()#['Year'].unique()


array(['Strategy TX:BASE', 'NDC', 'LEP', 'Conditional'], dtype=object)

In [5]:
# Let's use the NDC scenario as my baseline
total_gdp = drivers[(drivers['variable'] == 'gdp_mmm_usd') #& 
        #(drivers['primary_id'] == 74074) #&
        #(drivers['strategy'] == 'NDC')
        ][['Year','value','strategy']].sort_values(by=['strategy','Year'])
total_gdp.rename(columns={'value': 'gdp'}, inplace=True)      

In [6]:
total_gdp

,Year,gdp,strategy
452056,2015,116.940831,Conditional
452057,2016,112.487607,Conditional
452058,2017,121.473195,Conditional
452059,2018,129.305468,Conditional
452060,2019,112.565781,Conditional
...,...,...,...
319329,2046,163.219496,Strategy TX:BASE
319330,2047,166.483886,Strategy TX:BASE
319331,2048,169.813564,Strategy TX:BASE
319332,2049,173.209835,Strategy TX:BASE


In [7]:
emissions['strategy'].unique()

array(['Strategy TX:BASE', 'NDC', 'LEP', 'Conditional', 'Historical'],
      dtype=object)

In [8]:
# Use NDC emissions as baseline
total_emissions = emissions[(emissions['strategy'] != 'Historical')].groupby(['strategy','Year']).agg({'value': 'sum'}).reset_index()
total_emissions.rename(columns={'value': 'emissions'}, inplace=True) 

In [9]:
total_emissions

,strategy,Year,emissions
0,Conditional,2023,97.317200
1,Conditional,2024,93.740595
2,Conditional,2025,94.482377
3,Conditional,2026,95.251194
4,Conditional,2027,96.049087
...,...,...,...
107,Strategy TX:BASE,2046,112.851536
108,Strategy TX:BASE,2047,113.812774
109,Strategy TX:BASE,2048,114.791768
110,Strategy TX:BASE,2049,115.790317


In [10]:
df_final = pd.merge(total_emissions, total_gdp, on=['Year','strategy'], how='left')

# Deflactate for the value in 2023 from our data to the value in 2015 prices
#101162038920.033/55745806520.295 = 1.815
df_final['intensity'] = df_final['emissions']/df_final['gdp']*1.815

In [11]:
df_final

,strategy,Year,emissions,gdp,intensity
0,Conditional,2023,97.317200,101.162039,1.746018
1,Conditional,2024,93.740595,105.576742,1.611521
2,Conditional,2025,94.482377,107.688276,1.592425
3,Conditional,2026,95.251194,109.842042,1.573905
4,Conditional,2027,96.049087,112.038883,1.555970
...,...,...,...,...,...
107,Strategy TX:BASE,2046,112.851536,163.219496,1.254909
108,Strategy TX:BASE,2047,113.812774,166.483886,1.240782
109,Strategy TX:BASE,2048,114.791768,169.813564,1.226916
110,Strategy TX:BASE,2049,115.790317,173.209835,1.213323


In [12]:
df_final.to_csv(DATA_DIR_PATH.joinpath("emissions_intensity.csv"), index=False)